# Stanford Cars — BaCon baseline (paper mechanism)

This notebook adapts **BaCon / “Towards Distribution-Agnostic Generalized Category Discovery”** to Stanford Cars (196 classes) and is configured for **Kaggle T4 ×2** with **exactly 100 epochs**.

This is the comparison baseline: BaCon's original two-branch training, distribution estimation, regularization, and soft pseudo-label contrastive transfer, **without the repo's hard pseudo-label promotion extension**. The test set is evaluated exactly once after epoch 100, so there is no test-based checkpoint selection.

### Input / split protocol

You only need to attach the Kaggle dataset **`jutrera/stanford-car-dataset-by-classes-folder`**. The notebook consumes its `names.csv`, `anno_train.csv`, `anno_test.csv`, and class folders directly; no `.mat` files or additional dataset inputs are needed.

The note about `data_uq_idxs` is handled explicitly. Stanford Cars has no provided BaCon split files, so on the first run the notebook creates reproducible Cars equivalents: `l_k_uq_idxs.pt`, `unl_k_uq_idxs.pt`, and `unl_unk_uq_idxs.pt`, plus `manifest.json` and a portable zip under `/kaggle/working`. Unique IDs are stable annotation-row indices; class IDs come from the **official `names.csv` order**, not alphabetical folder order. The split follows the repo's CUB-style extension: seed 416, classes `0..97` known, `98..195` novel, 50% of each known class genuinely labeled, remaining known + all novel training images unlabeled. If you later attach a Kaggle Dataset containing these split files, the notebook auto-detects and reuses them rather than regenerating the partition.

**Before Run All:** set Kaggle `Accelerator → GPU T4 x2`, turn **Internet ON** (only for cloning the pinned repo and downloading DINO ViT-B/16 weights), attach the one dataset above, then Run All.


In [1]:
EXPERIMENT_NAME = 'cars_bacon_baseline_100e_2xT4'
ENABLE_MODE1 = False

In [2]:

# Kaggle setup: requires Accelerator = GPU T4 x2 and Internet ON for GitHub + DINO weights.
import os, sys, subprocess, random, json, math, time
from pathlib import Path

REPO_URL = "https://github.com/taamnguyeen04/AloGCD_FGLT.git"
REPO_COMMIT = "e71eac6cedf396ddcbb9ebc9c408c4a15218942e"
REPO_DIR = Path('/kaggle/working/AloGCD_FGLT')

import importlib.util
missing = []
for pip_name, module_name in [('scipy','scipy'), ('scikit-learn','sklearn'), ('pandas','pandas'), ('tqdm','tqdm')]:
    if importlib.util.find_spec(module_name) is None:
        missing.append(pip_name)
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
try:
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_COMMIT], check=True, capture_output=True)
except subprocess.CalledProcessError:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_COMMIT, '--depth', '1'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_COMMIT], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from PIL import Image
from scipy.io import loadmat
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from tqdm.auto import tqdm
from copy import deepcopy
from types import SimpleNamespace

from model.loss import DistillLoss, SupConLoss, SemiConLoss, info_nce_logits, get_params_groups
from model.vision_transformer import DINOHead

print('torch:', torch.__version__)
print('CUDA devices:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
assert torch.cuda.device_count() >= 2, 'This notebook is configured for Kaggle T4 x2. Select GPU T4 x2 first.'

torch.backends.cudnn.benchmark = True


Cloning into '/kaggle/working/AloGCD_FGLT'...


torch: 2.10.0+cu128
CUDA devices: 2
0 Tesla T4
1 Tesla T4


In [3]:

# ---- Fixed experiment protocol ----
SEED = 416
EPOCHS = 100
assert EPOCHS == 100
IMAGE_SIZE = 224
KNOWN_CLASSES = 98
TOTAL_CLASSES = 196
NOVEL_CLASSES = TOTAL_CLASSES - KNOWN_CLASSES
ANNO_RATIO = 0.50               # 50% of known-class train images are genuinely labeled
BATCH_SIZE = 128                # global batch; DataParallel splits it over 2 x T4
EVAL_BATCH_SIZE = 256
NUM_WORKERS = 4
GRAD_FROM_BLOCK = 11            # DINO ViT-B/16: only the last transformer block is branch-specific/trainable
LR = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-5
SUP_WEIGHT = 0.35
MEMAX_WEIGHT = 1.0
P = 1.1
ALPHA = 0.8
BETA = 0.5
TRO = 0.5
CE_WARMUP = 1
EST_FREQ = 10
PSEUDO_UPDATE_FREQ = 10
PSEUDO_TOP_RATIO = 0.80
MAX_SAMPLES_PER_CLASS = 500
MODE1_MIN_SUPPORT = 8
MODE1_CLASS_TOPK = 64
OUTPUT_DIR = Path('/kaggle/working') / EXPERIMENT_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print('Experiment:', EXPERIMENT_NAME)
print('Output:', OUTPUT_DIR)


Experiment: cars_bacon_baseline_100e_2xT4
Output: /kaggle/working/cars_bacon_baseline_100e_2xT4


In [4]:
# ---- Stanford Cars: ONLY Kaggle input required ----
# Dataset: jutrera/stanford-car-dataset-by-classes-folder
# This dataset already contains:
#   names.csv, anno_train.csv, anno_test.csv, and class-folder train/test images.
#
# IMPORTANT data_uq_idxs handling:
# The original BaCon repo stores fixed sample identities in three files for CIFAR:
#   l_k_uq_idxs.pt, unl_k_uq_idxs.pt, unl_unk_uq_idxs.pt
# Cars has no provided split files, so we create exact Cars-specific equivalents ONCE
# from stable official annotation-row uq_idx values, save them, and then load them.
# The split follows the repo's fine-grained CUB extension protocol:
#   official classes 0..97 = known, 98..195 = novel;
#   50% of each known class labeled, 50% unlabeled; all novel train images unlabeled;
#   seed = 416. No test labels are used to create the split.

import hashlib
from torchvision.datasets import ImageFolder

INPUT_ROOT = Path('/kaggle/input')

# Kaggle has used two mount layouts for dataset inputs.  The current UI can mount
# public datasets under /kaggle/input/datasets/<owner>/<slug>, while older notebooks
# often see /kaggle/input/<slug>.  Prefer the exact jutrera paths, then fall back to
# recursive metadata discovery.
preferred_roots = [
    Path('/kaggle/input/datasets/jutrera/stanford-car-dataset-by-classes-folder'),
    Path('/kaggle/input/stanford-car-dataset-by-classes-folder'),
]

def _looks_like_stanford_cars_root(p):
    return (
        p.is_dir()
        and (p / 'names.csv').is_file()
        and (p / 'anno_train.csv').is_file()
        and (p / 'anno_test.csv').is_file()
    )

DATASET_ROOT = next((p for p in preferred_roots if _looks_like_stanford_cars_root(p)), None)

if DATASET_ROOT is None:
    # Robust fallback: locate the metadata bundle anywhere below /kaggle/input.
    candidates = []
    for names_path in INPUT_ROOT.rglob('names.csv'):
        p = names_path.parent
        if _looks_like_stanford_cars_root(p):
            candidates.append(p)

    # De-duplicate and prefer a path containing the expected owner/slug.
    candidates = sorted(set(candidates), key=lambda p: (
        'jutrera' not in p.parts,
        'stanford-car-dataset-by-classes-folder' not in p.parts,
        len(p.parts),
        str(p),
    ))
    if not candidates:
        raise FileNotFoundError(
            'Could not find Stanford Cars metadata under /kaggle/input. Expected names.csv, '
            'anno_train.csv and anno_test.csv from jutrera/stanford-car-dataset-by-classes-folder.'
        )
    DATASET_ROOT = candidates[0]

print('Resolved Stanford Cars root:', DATASET_ROOT)

NAMES_CSV = DATASET_ROOT / 'names.csv'
TRAIN_ANNO_CSV = DATASET_ROOT / 'anno_train.csv'
TEST_ANNO_CSV = DATASET_ROOT / 'anno_test.csv'

# Kaggle mirrors of this dataset have appeared with either one or two car_data levels.
image_root_candidates = [
    DATASET_ROOT / 'car_data' / 'car_data',
    DATASET_ROOT / 'car_data',
]
IMAGE_ROOT = next((p for p in image_root_candidates if (p / 'train').is_dir() and (p / 'test').is_dir()), None)
if IMAGE_ROOT is None:
    # Last-resort discovery under this dataset only.
    pairs = []
    for t in DATASET_ROOT.rglob('train'):
        if t.is_dir() and (t.parent / 'test').is_dir():
            pairs.append(t.parent)
    if not pairs:
        raise FileNotFoundError('Could not find train/ and test/ class folders inside the attached Stanford Cars dataset.')
    IMAGE_ROOT = sorted(pairs, key=lambda p: len(p.parts))[0]

TRAIN_IMG_DIR = IMAGE_ROOT / 'train'
TEST_IMG_DIR = IMAGE_ROOT / 'test'

print('DATASET_ROOT =', DATASET_ROOT)
print('TRAIN_IMG_DIR=', TRAIN_IMG_DIR)
print('TEST_IMG_DIR =', TEST_IMG_DIR)
print('metadata      =', NAMES_CSV.name, TRAIN_ANNO_CSV.name, TEST_ANNO_CSV.name)

# Use names.csv as the canonical Stanford Cars class-ID order (1..196 in annotations).
# Do NOT use ImageFolder's alphabetical IDs for the GCD known/novel split.
class_names = pd.read_csv(NAMES_CSV, header=None, dtype=str).iloc[:, 0].str.strip().tolist()
assert len(class_names) == TOTAL_CLASSES, f'Expected 196 class names, found {len(class_names)}'
assert len(set(class_names)) == TOTAL_CLASSES
class_to_target = {name: i for i, name in enumerate(class_names)}

# Map official names.csv names to the actual Kaggle folder names.
# Some folder names sanitize characters such as '/' (e.g. "Ram C/V ...").
import re

def _norm_class_name(s):
    return re.sub(r'[^a-z0-9]+', '', str(s).lower())

def _build_folder_map(image_dir):
    out = {}
    for p in image_dir.iterdir():
        if p.is_dir():
            key = _norm_class_name(p.name)
            if key in out:
                raise RuntimeError(f'Duplicate normalized class folder key: {key}')
            out[key] = p.name
    return out

train_folder_map = _build_folder_map(TRAIN_IMG_DIR)
test_folder_map = _build_folder_map(TEST_IMG_DIR)

class_to_train_folder = {}
class_to_test_folder = {}
for name in class_names:
    key = _norm_class_name(name)
    if key not in train_folder_map or key not in test_folder_map:
        raise RuntimeError(f'Cannot map official class to Kaggle folder: {name}')
    class_to_train_folder[name] = train_folder_map[key]
    class_to_test_folder[name] = test_folder_map[key]


def load_records_from_csv(csv_path, image_dir):
    # Columns: filename, bbox_x1, bbox_y1, bbox_x2, bbox_y2, class_id (1-based)
    df = pd.read_csv(csv_path, header=None)
    if df.shape[1] < 6:
        raise RuntimeError(f'Unexpected annotation shape for {csv_path}: {df.shape}')
    records = []
    for uq_idx, row in df.iterrows():
        fname = str(row.iloc[0]).strip()
        target = int(row.iloc[5]) - 1
        if not (0 <= target < TOTAL_CLASSES):
            raise RuntimeError(f'Bad class id {target+1} in {csv_path}')
        class_name = class_names[target]
        folder_map = class_to_train_folder if image_dir == TRAIN_IMG_DIR else class_to_test_folder
        path = image_dir / folder_map[class_name] / fname
        if not path.is_file():
            raise FileNotFoundError(f'Annotation/image mismatch: {path}')
        records.append({'path': str(path), 'target': target, 'uq_idx': int(uq_idx)})
    return records


train_records = load_records_from_csv(TRAIN_ANNO_CSV, TRAIN_IMG_DIR)
test_records = load_records_from_csv(TEST_ANNO_CSV, TEST_IMG_DIR)
assert len(train_records) == 8144, f'Expected 8144 train images, got {len(train_records)}'
assert len(test_records) == 8041, f'Expected 8041 test images, got {len(test_records)}'
assert set(r['target'] for r in train_records) == set(range(TOTAL_CLASSES))
assert set(r['target'] for r in test_records) == set(range(TOTAL_CLASSES))
assert [r['uq_idx'] for r in train_records] == list(range(len(train_records)))
print(f'train={len(train_records)} test={len(test_records)} classes={TOTAL_CLASSES}')

# ---- Cars-specific data_uq_idxs split: reuse persistent .pt if attached, else generate ----
# First run needs ONLY the Stanford Cars dataset. We create the three BaCon-style split files
# in /kaggle/working and package them into a zip. Because /kaggle/working is ephemeral across
# Kaggle sessions, save that output as a Kaggle Dataset once if you want true cross-session reuse.
# On later runs, if an attached Kaggle input contains a compatible manifest.json + the 3 .pt files,
# this cell auto-detects and loads them instead of regenerating the split.
import shutil

SPLIT_NAME = f'cars196_k{KNOWN_CLASSES}_seed{SEED}_anno{int(ANNO_RATIO*100)}'
SPLIT_DIR = OUTPUT_DIR / 'data_uq_idxs' / SPLIT_NAME
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
L_K_FILE = SPLIT_DIR / 'l_k_uq_idxs.pt'
UNL_K_FILE = SPLIT_DIR / 'unl_k_uq_idxs.pt'
UNL_UNK_FILE = SPLIT_DIR / 'unl_unk_uq_idxs.pt'
MANIFEST_FILE = SPLIT_DIR / 'manifest.json'


def _compatible_split_dir(d):
    """Return parsed manifest if d is a compatible persistent split dataset, else None."""
    req = [d / 'l_k_uq_idxs.pt', d / 'unl_k_uq_idxs.pt', d / 'unl_unk_uq_idxs.pt', d / 'manifest.json']
    if not all(p.is_file() for p in req):
        return None
    try:
        m = json.loads((d / 'manifest.json').read_text())
    except Exception:
        return None
    if (
        m.get('seed') == SEED
        and m.get('known_classes') == [0, KNOWN_CLASSES - 1]
        and m.get('novel_classes') == [KNOWN_CLASSES, TOTAL_CLASSES - 1]
        and abs(float(m.get('annotation_ratio_known', -1)) - ANNO_RATIO) < 1e-12
        and m.get('uq_idx_source') == 'annotation CSV row index'
    ):
        return m
    return None


def find_persistent_split_input():
    # Search every Kaggle input, but never treat the raw Stanford Cars dataset itself as a split dataset.
    matches = []
    for manifest_path in INPUT_ROOT.rglob('manifest.json'):
        d = manifest_path.parent
        if DATASET_ROOT in d.parents or d == DATASET_ROOT:
            continue
        m = _compatible_split_dir(d)
        if m is not None:
            matches.append((d, m))
    if len(matches) > 1:
        # Prefer an exact split-name folder when several notebook outputs are attached.
        exact = [x for x in matches if x[0].name == SPLIT_NAME]
        matches = exact or matches
    return matches[0] if matches else (None, None)


def generate_cars_uq_split(out_dir):
    # Match the repo's CUB split style: np.random.seed(416), then per known class
    # np.random.choice(..., replace=False, floor(50%)). Test labels are never consulted.
    np.random.seed(SEED)
    targets = np.array([r['target'] for r in train_records], dtype=np.int64)
    uq_idxs = np.array([r['uq_idx'] for r in train_records], dtype=np.int64)

    l_k, unl_k, unl_unk = [], [], []
    for cls in range(TOTAL_CLASSES):
        cls_uq = uq_idxs[targets == cls]
        if cls < KNOWN_CLASSES:
            n_unlab = int((1.0 - ANNO_RATIO) * len(cls_uq))
            chosen_unlab = np.random.choice(cls_uq, replace=False, size=n_unlab)
            chosen_unlab_set = set(int(x) for x in chosen_unlab.tolist())
            chosen_lab = [int(x) for x in cls_uq.tolist() if int(x) not in chosen_unlab_set]
            l_k.extend(chosen_lab)
            unl_k.extend(int(x) for x in chosen_unlab.tolist())
        else:
            unl_unk.extend(int(x) for x in cls_uq.tolist())

    l_k = np.array(sorted(l_k), dtype=np.int64)
    unl_k = np.array(sorted(unl_k), dtype=np.int64)
    unl_unk = np.array(sorted(unl_unk), dtype=np.int64)

    out_dir.mkdir(parents=True, exist_ok=True)
    torch.save(torch.from_numpy(l_k), out_dir / 'l_k_uq_idxs.pt')
    torch.save(torch.from_numpy(unl_k), out_dir / 'unl_k_uq_idxs.pt')
    torch.save(torch.from_numpy(unl_unk), out_dir / 'unl_unk_uq_idxs.pt')


persistent_dir, persistent_manifest = find_persistent_split_input()
if persistent_dir is not None:
    print('Found persistent data_uq_idxs input:', persistent_dir)
    for name in ['l_k_uq_idxs.pt', 'unl_k_uq_idxs.pt', 'unl_unk_uq_idxs.pt', 'manifest.json']:
        shutil.copy2(persistent_dir / name, SPLIT_DIR / name)
    split_origin = f'loaded from Kaggle input: {persistent_dir}'
else:
    if not (L_K_FILE.exists() and UNL_K_FILE.exists() and UNL_UNK_FILE.exists()):
        generate_cars_uq_split(SPLIT_DIR)
    split_origin = 'generated deterministically in this Kaggle session'

l_k_uq_idxs = torch.load(L_K_FILE, map_location='cpu').long().numpy()
unl_k_uq_idxs = torch.load(UNL_K_FILE, map_location='cpu').long().numpy()
unl_unk_uq_idxs = torch.load(UNL_UNK_FILE, map_location='cpu').long().numpy()

# Strong split invariants: no overlap, complete partition, and correct known/novel membership.
all_train_uq = set(range(len(train_records)))
L = set(map(int, l_k_uq_idxs.tolist()))
UK = set(map(int, unl_k_uq_idxs.tolist()))
UU = set(map(int, unl_unk_uq_idxs.tolist()))
assert L.isdisjoint(UK) and L.isdisjoint(UU) and UK.isdisjoint(UU)
assert L | UK | UU == all_train_uq
assert all(train_records[i]['target'] < KNOWN_CLASSES for i in L)
assert all(train_records[i]['target'] < KNOWN_CLASSES for i in UK)
assert all(train_records[i]['target'] >= KNOWN_CLASSES for i in UU)

split_bytes = b''.join([
    np.asarray(l_k_uq_idxs, dtype=np.int64).tobytes(),
    np.asarray(unl_k_uq_idxs, dtype=np.int64).tobytes(),
    np.asarray(unl_unk_uq_idxs, dtype=np.int64).tobytes(),
])
split_sha256 = hashlib.sha256(split_bytes).hexdigest()

# If we loaded an existing manifest, its hash must agree with the actual .pt contents.
if persistent_manifest is not None and persistent_manifest.get('partition_sha256'):
    assert persistent_manifest['partition_sha256'] == split_sha256, 'Persistent split hash mismatch'

manifest = {
    'dataset': 'jutrera/stanford-car-dataset-by-classes-folder',
    'split_name': SPLIT_NAME,
    'class_order_source': 'names.csv (official Stanford Cars 1-based class order -> 0-based)',
    'uq_idx_source': 'annotation CSV row index',
    'seed': SEED,
    'known_classes': [0, KNOWN_CLASSES - 1],
    'novel_classes': [KNOWN_CLASSES, TOTAL_CLASSES - 1],
    'annotation_ratio_known': ANNO_RATIO,
    'l_k_count': int(len(l_k_uq_idxs)),
    'unl_k_count': int(len(unl_k_uq_idxs)),
    'unl_unk_count': int(len(unl_unk_uq_idxs)),
    'partition_sha256': split_sha256,
    'test_used_for_split': False,
}
with open(MANIFEST_FILE, 'w') as f:
    json.dump(manifest, f, indent=2)

# Produce a tiny portable artifact you can save as a Kaggle Dataset after the first run.
SPLIT_ZIP_BASE = Path('/kaggle/working') / f'stanford_cars_gcd_uq_splits_{SPLIT_NAME}'
SPLIT_ZIP = Path(shutil.make_archive(str(SPLIT_ZIP_BASE), 'zip', root_dir=SPLIT_DIR))

real_labeled_indices = sorted(map(int, l_k_uq_idxs.tolist()))
unlabeled_indices = sorted(map(int, np.concatenate([unl_k_uq_idxs, unl_unk_uq_idxs]).tolist()))

print('\nSplit origin:', split_origin)
print('Persistent split directory:', SPLIT_DIR)
print('Portable split zip:', SPLIT_ZIP)
print('For future Kaggle sessions: save/publish this split folder/zip as a Kaggle Dataset, then attach it; the notebook will auto-load it.')
print('\nCars GCD split (repo-style uq_idx partitions)')
print(json.dumps(manifest, indent=2))
print('first known class:', 0, class_names[0])
print('last known class :', KNOWN_CLASSES - 1, class_names[KNOWN_CLASSES - 1])
print('first novel class:', KNOWN_CLASSES, class_names[KNOWN_CLASSES])
print('last novel class :', TOTAL_CLASSES - 1, class_names[-1])

mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)
resize_size = int(IMAGE_SIZE / 0.875)
train_transform = transforms.Compose([
    transforms.Resize(resize_size, InterpolationMode.BICUBIC),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])
test_transform = transforms.Compose([
    transforms.Resize(resize_size, InterpolationMode.BICUBIC),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])


class CarsPoolDataset(Dataset):
    def __init__(self, records, items, transform, two_views=False):
        self.records = records
        self.items = items  # (uq_idx, training_label, is_labeled)
        self.transform = transform
        self.two_views = two_views

    def __len__(self):
        return len(self.items)

    def __getitem__(self, j):
        uq_idx, train_label, is_labeled = self.items[j]
        r = self.records[uq_idx]
        assert r['uq_idx'] == uq_idx
        with open(r['path'], 'rb') as f:
            img = Image.open(f).convert('RGB')
        if self.two_views:
            x = [self.transform(img), self.transform(img)]
        else:
            x = self.transform(img)
        return x, int(train_label), int(uq_idx), np.array([1 if is_labeled else 0], dtype=np.int64)


class CarsTestDataset(Dataset):
    def __init__(self, records, transform):
        self.records = records
        self.transform = transform
    def __len__(self):
        return len(self.records)
    def __getitem__(self, i):
        r = self.records[i]
        with open(r['path'], 'rb') as f:
            img = Image.open(f).convert('RGB')
        return self.transform(img), int(r['target']), int(r['uq_idx'])


class TrainPool:
    def __init__(self, real_labeled, base_unlabeled):
        self.real_labeled = set(real_labeled)
        self.base_unlabeled = set(base_unlabeled)
        self.promoted = {}  # uq_idx -> pseudo label

    @property
    def active_unlabeled(self):
        return sorted(self.base_unlabeled - set(self.promoted.keys()))

    def make_items(self):
        # Keep labeled items first, matching the original MergedDataset weighting logic.
        labeled = [(i, train_records[i]['target'], True) for i in sorted(self.real_labeled)]
        labeled += [(i, int(y), True) for i, y in sorted(self.promoted.items())]
        # Ground truth of every active unlabeled sample is deliberately replaced by -1.
        unlabeled = [(i, -1, False) for i in self.active_unlabeled]
        return labeled + unlabeled, len(labeled), len(unlabeled)

    def promote(self, selected):
        # selected: iterable of (uq_idx, pseudo_label)
        added = 0
        for idx, y in selected:
            if idx in self.base_unlabeled and idx not in self.promoted:
                self.promoted[int(idx)] = int(y)
                added += 1
        return added


pool = TrainPool(real_labeled_indices, unlabeled_indices)
test_dataset = CarsTestDataset(test_records, test_transform)


def make_train_loader(pool):
    items, n_lab, n_unlab = pool.make_items()
    ds = CarsPoolDataset(train_records, items, train_transform, two_views=True)
    if n_unlab > 0:
        weights = [1.0] * n_lab + [n_lab / n_unlab] * n_unlab
    else:
        weights = [1.0] * n_lab
    sampler = WeightedRandomSampler(torch.DoubleTensor(weights), num_samples=len(ds), replacement=True)
    return DataLoader(ds, batch_size=BATCH_SIZE, sampler=sampler, shuffle=False, drop_last=True,
                      num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=(NUM_WORKERS > 0))


def make_pool_eval_loader(pool):
    items, _, _ = pool.make_items()
    ds = CarsPoolDataset(train_records, items, test_transform, two_views=False)
    return DataLoader(ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                      pin_memory=True, persistent_workers=(NUM_WORKERS > 0))


def make_unlabeled_selection_loader(pool):
    items = [(i, -1, False) for i in pool.active_unlabeled]
    ds = CarsPoolDataset(train_records, items, train_transform, two_views=True)
    return DataLoader(ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                      pin_memory=True, persistent_workers=(NUM_WORKERS > 0))


Resolved Stanford Cars root: /kaggle/input/datasets/jutrera/stanford-car-dataset-by-classes-folder
DATASET_ROOT = /kaggle/input/datasets/jutrera/stanford-car-dataset-by-classes-folder
TRAIN_IMG_DIR= /kaggle/input/datasets/jutrera/stanford-car-dataset-by-classes-folder/car_data/car_data/train
TEST_IMG_DIR = /kaggle/input/datasets/jutrera/stanford-car-dataset-by-classes-folder/car_data/car_data/test
metadata      = names.csv anno_train.csv anno_test.csv
train=8144 test=8041 classes=196

Split origin: generated deterministically in this Kaggle session
Persistent split directory: /kaggle/working/cars_bacon_baseline_100e_2xT4/data_uq_idxs/cars196_k98_seed416_anno50
Portable split zip: /kaggle/working/stanford_cars_gcd_uq_splits_cars196_k98_seed416_anno50.zip
For future Kaggle sessions: save/publish this split folder/zip as a Kaggle Dataset, then attach it; the notebook will auto-load it.

Cars GCD split (repo-style uq_idx partitions)
{
  "dataset": "jutrera/stanford-car-dataset-by-classes-f

In [5]:

# ---- BaCon dual-branch model, adapted to actually use both Kaggle T4s ----
# Original repo hard-codes cuda:0. Here DataParallel runs the frozen DINO prefix and
# the two branch tails over GPU 0 + GPU 1. Only the final transformer block is trainable.

class CEHead(nn.Module):
    def __init__(self, in_dim, out_dim, norm_last_layer=True):
        super().__init__()
        self.last_layer = nn.utils.weight_norm(nn.Linear(in_dim, out_dim, bias=False))
        self.last_layer.weight_g.data.fill_(1)
        if norm_last_layer:
            self.last_layer.weight_g.requires_grad = False
    def forward(self, x):
        return self.last_layer(F.normalize(x, dim=-1, p=2))


class TailBranch(nn.Module):
    def __init__(self, blocks, norm, head):
        super().__init__()
        self.blocks = nn.ModuleList(blocks)
        self.norm = norm
        self.head = head
    def forward(self, x, with_head=True):
        for blk in self.blocks:
            x = blk(x)
        feat = self.norm(x)[:, 0]
        if with_head:
            return feat, self.head(feat)
        return feat


class BaConDual(nn.Module):
    def __init__(self, pretrained_dino, num_classes, grad_from_block=11):
        super().__init__()
        # Reproduce BaCon's shared frozen prefix + duplicated trainable tail blocks.
        for p in pretrained_dino.parameters():
            p.requires_grad = False
        for name, p in pretrained_dino.named_parameters():
            if 'blocks.' in name:
                block_num = int(name.split('blocks.')[1].split('.')[0])
                if block_num >= grad_from_block:
                    p.requires_grad = True

        ce_tail = [deepcopy(pretrained_dino.blocks[i]) for i in range(grad_from_block, len(pretrained_dino.blocks))]
        cl_tail = [deepcopy(pretrained_dino.blocks[i]) for i in range(grad_from_block, len(pretrained_dino.blocks))]
        ce_norm = deepcopy(pretrained_dino.norm)
        cl_norm = deepcopy(pretrained_dino.norm)
        ce_head = CEHead(768, num_classes)
        cl_head = DINOHead(in_dim=768, out_dim=65536, nlayers=3)

        # Prefix is frozen. Its original tail blocks are retained but never called.
        self.shared = pretrained_dino
        for p in self.shared.parameters():
            p.requires_grad = False
        self.grad_from_block = grad_from_block
        self.ce_branch = TailBranch(ce_tail, ce_norm, ce_head)
        self.cl_branch = TailBranch(cl_tail, cl_norm, cl_head)

    def _prefix(self, images):
        # No backward graph through the frozen shared prefix.
        with torch.no_grad():
            x = self.shared.prepare_tokens(images)
            for i in range(self.grad_from_block):
                x = self.shared.blocks[i](x)
        return x

    @torch.autocast(device_type='cuda', dtype=torch.float16)
    def forward(self, images, mode='train'):
        x = self._prefix(images)
        if mode == 'train':
            ce_feat, ce_logits = self.ce_branch(x, with_head=True)
            cl_feat, cl_proj = self.cl_branch(x, with_head=True)
            return ce_logits, cl_proj
        if mode == 'ce':
            _, ce_logits = self.ce_branch(x, with_head=True)
            return ce_logits
        if mode == 'cl_feature':
            return self.cl_branch(x, with_head=False)
        raise ValueError(mode)


def build_model():
    print('Loading DINO ViT-B/16 pretrained backbone...')
    dino = torch.hub.load('facebookresearch/dino:main', 'dino_vitb16', trust_repo=True)
    core = BaConDual(dino, TOTAL_CLASSES, GRAD_FROM_BLOCK).cuda(0)
    model = nn.DataParallel(core, device_ids=[0, 1], output_device=0)
    trainable = sum(p.numel() for p in core.parameters() if p.requires_grad)
    total = sum(p.numel() for p in core.parameters())
    print(f'trainable params={trainable/1e6:.2f}M / total={total/1e6:.2f}M')
    return model


def compute_reg_loss(student_out, est_count):
    avg_probs = (student_out / 0.1).softmax(dim=1).mean(dim=0)
    avg_probs = avg_probs * 1.0 / (est_count ** P)
    avg_probs = avg_probs / avg_probs.sum()
    return -torch.sum(torch.log(avg_probs ** (-avg_probs))) + math.log(float(len(avg_probs)))


def compute_softconloss(student_out, cl_proj_feature, sup_con_labels, sup_cl_proj_feature, mask_lab, args):
    # Same self-balanced soft pseudo-label transfer as BaCon, generalized to 196 classes.
    logits = (student_out / 0.1) - args.est_adjustment
    view1_probs, view2_probs = logits.softmax(dim=1).chunk(2)
    soft_labels = (view1_probs + view2_probs) / 2

    known_class_sampling_rate = ((1 / args.est_dist) * args.est_dist.min()) ** ALPHA
    existing_class_idx = torch.unique(sup_con_labels)
    sampling_rate = ((1 / args.est_dist) * args.est_dist.min()) ** BETA
    sampling_rate[existing_class_idx] = known_class_sampling_rate[existing_class_idx]

    batch_confidence, batch_preds = soft_labels[~mask_lab].max(dim=1)
    sampling_mask = torch.zeros(len(soft_labels), dtype=torch.bool, device=soft_labels.device)
    all_ids = torch.arange(len(soft_labels), device=soft_labels.device)
    for cls_idx in torch.unique(batch_preds):
        cls_mask = (batch_preds == cls_idx)
        cls_ins_num = int(cls_mask.sum().item())
        if cls_ins_num == 0:
            continue
        prob = float(sampling_rate[cls_idx].clamp(0, 1).item())
        cls_sample_num = int(torch.bernoulli(torch.full((cls_ins_num,), prob, device=soft_labels.device)).sum().item())
        if cls_sample_num <= 0:
            continue
        cls_conf = batch_confidence[cls_mask]
        _, local_idx = torch.topk(cls_conf, k=min(cls_sample_num, cls_ins_num))
        sampling_mask[all_ids[~mask_lab][cls_mask][local_idx]] = True

    semicon_soft_labels = soft_labels[(~mask_lab) & sampling_mask]
    semicon_feats = torch.cat([
        f[(~mask_lab) & sampling_mask].unsqueeze(1) for f in cl_proj_feature.chunk(2)
    ], dim=1)
    sup_labels_soft = soft_labels[mask_lab]
    all_con_feats = torch.cat([sup_cl_proj_feature, semicon_feats], dim=0)
    all_soft_labels = torch.cat([sup_labels_soft, semicon_soft_labels], dim=0)
    soft_mask = (all_soft_labels.unsqueeze(1) * all_soft_labels.unsqueeze(0)).sum(dim=2)
    soft_mask[torch.eye(len(soft_mask), device=soft_mask.device, dtype=torch.bool)] = 1
    return SemiConLoss(args=args)(all_con_feats, soft_mask=soft_mask.detach())


@torch.no_grad()
def estimate_distribution(model, pool, args):
    """BaCon distribution estimate using only train data; unlabeled GT is never used."""
    model.eval()
    feats, labels, masks = [], [], []
    for images, y, _, mask in tqdm(make_pool_eval_loader(pool), desc='dist-est', leave=False):
        images = images.cuda(0, non_blocking=True)
        f = model(images, mode='cl_feature')
        feats.append(F.normalize(f.float(), dim=-1).cpu().numpy())
        labels.append(y.numpy())
        masks.append(mask.view(-1).bool().numpy())
    feats = np.concatenate(feats)
    labels = np.concatenate(labels).astype(int)
    masks = np.concatenate(masks).astype(bool)

    km = KMeans(n_clusters=TOTAL_CLASSES, random_state=0, n_init=1, max_iter=100).fit(feats)
    clusters = km.labels_
    cluster_counts = np.bincount(clusters, minlength=TOTAL_CLASSES).astype(np.float32)

    # Align only known labels using the genuine labeled subset.
    w = np.zeros((TOTAL_CLASSES, KNOWN_CLASSES), dtype=np.int64)
    for cl, y in zip(clusters[masks], labels[masks]):
        if 0 <= y < KNOWN_CLASSES:
            w[cl, y] += 1
    row_ind, col_ind = linear_sum_assignment(w.max() - w)
    known_to_cluster = {int(c): int(r) for r, c in zip(row_ind, col_ind)}

    est = np.ones(TOTAL_CLASSES, dtype=np.float32)
    used_clusters = set()
    for c in range(KNOWN_CLASSES):
        cl = known_to_cluster.get(c)
        if cl is not None:
            est[c] = max(cluster_counts[cl], 1.0)
            used_clusters.add(cl)
    remaining = [cluster_counts[c] for c in range(TOTAL_CLASSES) if c not in used_clusters]
    remaining = sorted(remaining, reverse=True)
    for j, c in enumerate(range(KNOWN_CLASSES, TOTAL_CLASSES)):
        est[c] = max(remaining[j] if j < len(remaining) else 1.0, 1.0)

    args.est_dist = torch.tensor(est, dtype=torch.float32, device='cuda:0')
    freq = args.est_dist / args.est_dist.sum()
    args.est_adjustment = torch.log(freq.pow(TRO) + 1e-12)
    return args.est_dist


@torch.no_grad()
def final_cluster_eval(model):
    """Final-only test evaluation. Test labels are never visible to training or Mode 1."""
    model.eval()
    loader = DataLoader(test_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=(NUM_WORKERS > 0))
    feats, ys = [], []
    for images, y, _ in tqdm(loader, desc='final test features'):
        images = images.cuda(0, non_blocking=True)
        f = model(images, mode='cl_feature')
        feats.append(F.normalize(f.float(), dim=-1).cpu().numpy())
        ys.append(y.numpy())
    feats = np.concatenate(feats)
    y_true = np.concatenate(ys).astype(int)
    pred_cluster = KMeans(n_clusters=TOTAL_CLASSES, random_state=0, n_init=10).fit_predict(feats)
    w = np.zeros((TOTAL_CLASSES, TOTAL_CLASSES), dtype=np.int64)
    for p, y in zip(pred_cluster, y_true):
        w[p, y] += 1
    row_ind, col_ind = linear_sum_assignment(w.max() - w)
    mapping = {int(r): int(c) for r, c in zip(row_ind, col_ind)}
    y_pred = np.array([mapping[int(p)] for p in pred_cluster])
    old = y_true < KNOWN_CLASSES
    new = ~old
    metrics = {
        'all_acc': float((y_pred == y_true).mean() * 100),
        'old_acc': float((y_pred[old] == y_true[old]).mean() * 100),
        'new_acc': float((y_pred[new] == y_true[new]).mean() * 100),
    }
    return metrics


In [6]:

# ---- Leakage-free Mode 1 target selection ----
# Original repo picks the target class from per-class *test accuracy* -> test leakage.
# Here target-class reliability uses ONLY active unlabeled TRAIN samples:
#   1) two stochastic views must predict the same known class;
#   2) class score = mean of its top-K two-view averaged confidences;
#   3) promote top 80% within the selected class (cap 500), same spirit as repo Mode 1.
# Promoted samples are removed from the active unlabeled pool and are re-augmented from
# the original image every epoch (no cached transformed tensor duplicates).

@torch.no_grad()
def leakage_free_mode1_update(model, pool, epoch):
    model.eval()
    per_class = {c: [] for c in range(KNOWN_CLASSES)}
    loader = make_unlabeled_selection_loader(pool)

    for views, _, uq_idx, _ in tqdm(loader, desc=f'mode1-select@{epoch}', leave=False):
        images = torch.cat(views, dim=0).cuda(0, non_blocking=True)
        logits = model(images, mode='ce').float()
        p1, p2 = F.softmax(logits, dim=1).chunk(2)
        avg = (p1 + p2) / 2
        pred1 = p1.argmax(dim=1)
        pred2 = p2.argmax(dim=1)
        conf, pred = avg.max(dim=1)
        agree = (pred1 == pred2) & (pred < KNOWN_CLASSES)

        for i in torch.where(agree)[0].tolist():
            c = int(pred[i].item())
            per_class[c].append((float(conf[i].item()), int(uq_idx[i].item())))

    eligible = {c: vals for c, vals in per_class.items() if len(vals) >= MODE1_MIN_SUPPORT}
    if not eligible:
        print(f'[Mode1 @ epoch {epoch}] no class has >= {MODE1_MIN_SUPPORT} agreeing samples; skip')
        return {'epoch': epoch, 'skipped': True, 'reason': 'insufficient_support'}

    class_scores = {}
    for c, vals in eligible.items():
        top = sorted((s for s, _ in vals), reverse=True)[:MODE1_CLASS_TOPK]
        class_scores[c] = float(np.mean(top))
    target_class = max(class_scores, key=class_scores.get)

    candidates = sorted(eligible[target_class], key=lambda x: x[0], reverse=True)
    n_keep = min(int(len(candidates) * PSEUDO_TOP_RATIO), MAX_SAMPLES_PER_CLASS)
    if n_keep <= 0:
        return {'epoch': epoch, 'skipped': True, 'reason': 'zero_keep'}
    selected = [(idx, target_class) for conf, idx in candidates[:n_keep]]
    added = pool.promote(selected)

    event = {
        'epoch': epoch,
        'skipped': False,
        'target_class': int(target_class),
        'class_reliability': float(class_scores[target_class]),
        'agreeing_candidates': int(len(candidates)),
        'promoted': int(added),
        'mean_selected_conf': float(np.mean([c for c, _ in candidates[:n_keep]])),
        'active_unlabeled_after': int(len(pool.active_unlabeled)),
        'total_promoted': int(len(pool.promoted)),
    }
    print('[Mode1]', event)
    return event


def offline_pseudo_audit(pool):
    """Called only AFTER all 100 epochs; diagnostic GT cannot affect training."""
    if not pool.promoted:
        return {'n': 0, 'correct': 0, 'wrong_old': 0, 'novel_contamination': 0, 'precision': None}
    correct = wrong_old = novel = 0
    for idx, pseudo_y in pool.promoted.items():
        true_y = train_records[idx]['target']
        if true_y >= KNOWN_CLASSES:
            novel += 1
        elif true_y == pseudo_y:
            correct += 1
        else:
            wrong_old += 1
    n = len(pool.promoted)
    return {
        'n': n,
        'correct': correct,
        'wrong_old': wrong_old,
        'novel_contamination': novel,
        'precision': correct / n if n else None,
    }


In [7]:

# ---- Train exactly 100 epochs; test set is evaluated exactly once, after training ----
args = SimpleNamespace(
    n_views=2,
    warmup_teacher_temp_epochs=30,
    warmup_teacher_temp=0.07,
    teacher_temp=0.04,
)

model = build_model()
core = model.module
param_groups = get_params_groups(core.ce_branch) + get_params_groups(core.cl_branch)
optimizer = torch.optim.SGD(param_groups, lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 1e-3)
cluster_criterion = DistillLoss(
    warmup_teacher_temp_epochs=args.warmup_teacher_temp_epochs,
    nepochs=EPOCHS,
    ncrops=2,
    warmup_teacher_temp=args.warmup_teacher_temp,
    teacher_temp=args.teacher_temp,
    args=args,
)
scaler = torch.cuda.amp.GradScaler(enabled=True)

est_count = estimate_distribution(model, pool, args)
history = []
mode1_events = []

for e0 in range(EPOCHS):
    epoch = e0 + 1
    loader = make_train_loader(pool)
    model.train()
    loss_ce_sum = loss_cl_sum = total_sum = 0.0
    n_steps = 0

    pbar = tqdm(loader, desc=f'Epoch {epoch:03d}/{EPOCHS}')
    for views, class_labels, _, mask_lab in pbar:
        mask_lab = mask_lab.view(-1).cuda(0, non_blocking=True).bool()
        class_labels = class_labels.cuda(0, non_blocking=True)
        images = torch.cat(views, dim=0).cuda(0, non_blocking=True)
        if not mask_lab.any():
            continue

        optimizer.zero_grad(set_to_none=True)
        student_out, cl_proj = model(images, mode='train')
        # Keep numerically sensitive losses in fp32 while retaining AMP speed in the model.
        student_out = student_out.float()
        cl_proj = F.normalize(cl_proj.float(), dim=-1)
        teacher_out = student_out.detach()

        cluster_loss = cluster_criterion(student_out, teacher_out, e0)
        sup_logits = torch.cat([f[mask_lab] for f in (student_out / 0.1).chunk(2)], dim=0)
        sup_labels = torch.cat([class_labels[mask_lab] for _ in range(2)], dim=0)
        cls_loss = F.cross_entropy(sup_logits, sup_labels)
        me_max_loss = compute_reg_loss(student_out, est_count)
        cluster_loss = cluster_loss + MEMAX_WEIGHT * me_max_loss
        loss_ce = (1 - SUP_WEIGHT) * cluster_loss + SUP_WEIGHT * cls_loss

        contrastive_logits, contrastive_labels = info_nce_logits(cl_proj)
        contrastive_loss = F.cross_entropy(contrastive_logits, contrastive_labels)
        sup_cl_proj = torch.cat([f[mask_lab].unsqueeze(1) for f in cl_proj.chunk(2)], dim=1)
        sup_con_labels = class_labels[mask_lab]
        sup_con_loss = SupConLoss()(sup_cl_proj, labels=sup_con_labels)

        if e0 >= CE_WARMUP:
            soft_con_loss = compute_softconloss(
                student_out, cl_proj, sup_con_labels, sup_cl_proj, mask_lab, args
            )
            loss_cl = ((1 - SUP_WEIGHT) * contrastive_loss + (SUP_WEIGHT / 2) * sup_con_loss
                       + (SUP_WEIGHT / 2) * soft_con_loss)
        else:
            soft_con_loss = torch.tensor(float('nan'), device='cuda:0')
            loss_cl = (1 - SUP_WEIGHT) * contrastive_loss + SUP_WEIGHT * sup_con_loss

        total_loss = loss_ce + loss_cl
        scaler.scale(total_loss).backward()
        scaler.step(optimizer)
        scaler.update()

        n_steps += 1
        loss_ce_sum += float(loss_ce.detach())
        loss_cl_sum += float(loss_cl.detach())
        total_sum += float(total_loss.detach())
        pbar.set_postfix(loss=f'{total_sum/n_steps:.3f}', lab=int(mask_lab.sum()), pseudo=len(pool.promoted))

    scheduler.step()
    row = {
        'epoch': epoch,
        'loss_ce': loss_ce_sum / max(n_steps, 1),
        'loss_cl': loss_cl_sum / max(n_steps, 1),
        'loss_total': total_sum / max(n_steps, 1),
        'lr': optimizer.param_groups[0]['lr'],
        'n_promoted': len(pool.promoted),
        'active_unlabeled': len(pool.active_unlabeled),
    }
    history.append(row)
    pd.DataFrame(history).to_csv(OUTPUT_DIR / 'train_history.csv', index=False)

    # Paper mechanism: refresh distribution estimate periodically using train data only.
    if epoch % EST_FREQ == 0 and epoch < EPOCHS:
        est_count = estimate_distribution(model, pool, args)

    # Extra Mode 1 mechanism, only in the Mode 1 notebook.
    if ENABLE_MODE1 and epoch % PSEUDO_UPDATE_FREQ == 0 and epoch < EPOCHS:
        event = leakage_free_mode1_update(model, pool, epoch)
        mode1_events.append(event)
        pd.DataFrame(mode1_events).to_csv(OUTPUT_DIR / 'mode1_events.csv', index=False)

    if epoch % 10 == 0:
        torch.save({'epoch': epoch, 'model': core.state_dict(), 'promoted': dict(pool.promoted)},
                   OUTPUT_DIR / f'checkpoint_epoch{epoch:03d}.pt')

# Fixed final checkpoint: no best-test-epoch selection.
torch.save({'epoch': EPOCHS, 'model': core.state_dict(), 'promoted': dict(pool.promoted)},
           OUTPUT_DIR / 'final_epoch100.pt')

print('\nRunning the ONLY test-set evaluation now (after all 100 epochs)...')
final_metrics = final_cluster_eval(model)
with open(OUTPUT_DIR / 'final_metrics.json', 'w') as f:
    json.dump(final_metrics, f, indent=2)
print('FINAL:', final_metrics)

if ENABLE_MODE1:
    audit = offline_pseudo_audit(pool)
    with open(OUTPUT_DIR / 'offline_pseudo_audit.json', 'w') as f:
        json.dump(audit, f, indent=2)
    print('OFFLINE pseudo audit (computed only after training):', audit)


Loading DINO ViT-B/16 pretrained backbone...
Downloading: "https://github.com/facebookresearch/dino/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/dino/dino_vitbase16_pretrain/dino_vitbase16_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dino_vitbase16_pretrain.pth


100%|██████████| 327M/327M [00:01<00:00, 318MB/s]
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


trainable params=37.40M / total=123.27M


/tmp/ipykernel_23/1831382944.py:22: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=True)


dist-est:   0%|          | 0/32 [00:00<?, ?it/s]

...

Epoch 099/100:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 100/100:   0%|          | 0/63 [00:00<?, ?it/s]


Running the ONLY test-set evaluation now (after all 100 epochs)...


final test features:   0%|          | 0/32 [00:00<?, ?it/s]

FINAL: {'all_acc': 42.24598930481284, 'old_acc': 49.97501249375313, 'new_acc': 34.58776924981431}


In [8]:
print('Split SHA256:', split_sha256)

# Compact result view
print('Experiment:', EXPERIMENT_NAME)
print(pd.DataFrame(history).tail(10).to_string(index=False))
print('\nFinal clustering metrics (%):')
print(json.dumps(final_metrics, indent=2))
if ENABLE_MODE1 and mode1_events:
    print('\nMode 1 events:')
    display(pd.DataFrame(mode1_events))
print('\nArtifacts in', OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.iterdir()):
    print(' -', p.name)


Split SHA256: 45c9e537b94c2c9133190b431daab33854e87c67de066bdb315ed5d7d132dfa2
Experiment: cars_bacon_baseline_100e_2xT4
 epoch  loss_ce  loss_cl  loss_total       lr  n_promoted  active_unlabeled
    91 0.519322 3.357822    3.877144 0.002083           0              6090
    92 0.510610 3.349029    3.859639 0.001669           0              6090
    93 0.519992 3.362672    3.882663 0.001303           0              6090
    94 0.519521 3.356800    3.876321 0.000985           0              6090
    95 0.518850 3.348599    3.867449 0.000715           0              6090
    96 0.518848 3.348840    3.867688 0.000494           0              6090
    97 0.512144 3.355164    3.867308 0.000322           0              6090
    98 0.515330 3.348258    3.863589 0.000199           0              6090
    99 0.516131 3.339568    3.855699 0.000125           0              6090
   100 0.517232 3.349745    3.866977 0.000100           0              6090

Final clustering metrics (%):
{
  "all_acc